# Road anomaly detection
## Business understanding, data understanding, and data preparation

This notebook is the working version of the written report in `reports/01_business_and_data_understanding.md`. It checks the RoadAnomaly files, shows how each photograph lines up with its mask, and verifies the training, validation, and test sets.

The prepared images are built by `python scripts/run_exploration_and_split.py`. Run that script once before the later cells if `data/processed` is not already present.

**What to tell the professor, in order.** Each line is one figure.

1. **Figure 11, then Figure 5.** Every photograph has a pixel mask. In the file, 0 is the ordinary scene and 2 is the obstacle. The yellow line is the edge of that mask, and it follows the object, including a tire that covers 0.37% of the frame.
2. **Figure 4.** Obstacles are 9.8% of pixels. A model that predicts background everywhere is about 90% accurate and finds nothing, so the later scores are precision, recall, F1, IoU, and Dice.
3. **Figures 6 and 3.** The same task includes that tire and a herd covering 42% of the image. Cones and loose tires are the small objects.
4. **Figure 10.** Eleven frames are repeated views of three events. They all stay in training, so the test score cannot come from a near-copy.
5. **Figures 8 and 7.** Each 1280×720 frame is padded to 512×512 without stretching the road. The split is 42 / 9 / 9.

## 1. Business understanding

Usual road detectors are trained on common classes such as car, pedestrian, and road. RoadAnomaly, from Lis, Nakka, Fua, and Salzmann (ICCV 2019), is a set of scenes those classes do not cover. A miss matters to the people in the vehicle and to anyone in the lane. A warning or a planner can act on a located region.

**Business objectives**

- Detect the unexpected obstacle.
- Locate it in the camera image.
- Cover different obstacle types and very different sizes.
- Produce a region a person can compare with the annotation.

The original paper trains on ordinary Cityscapes images and tests on these 60. That protocol does not use the anomaly masks for training. This module provides the masks and asks for a train, validation, and test organisation, so the task prepared here is supervised segmentation. Numbers from our split are for this project. A comparison with the paper would test a model on all 60 images without training on them.

## 2. Data-science objectives

| Official value | Meaning | Stored training mask |
| --- | --- | --- |
| 0 | Background | 0 |
| 2 | Obstacle | 255, and any pixel above 0 is the positive class |

The filename prefix (`animals`, `cones`, `obstacles`, `lost_tires`, `vehicle`) balances the split. It is not the class the network should predict. One image can contain several unexpected objects, and the mask marks all of them.

**Objectives of this notebook**

1. Prove the image–mask pairing: same size, values restricted to 0 and 2.
2. Measure how rare the obstacle pixels are, and how much that share varies.
3. Confirm the 42 / 9 / 9 split keeps each repeated event inside one subset.
4. Confirm the 512×512 files still contain every obstacle.

Pixel accuracy is the wrong headline number: background is 90.2% of pixels. The scores that match the business need are precision, recall, F1, IoU, and Dice.

## 3. Data understanding

The next cell loads all 60 pairs. It raises an error if a listed image has no mask, if the sizes differ, or if a mask contains a value other than 0 or 2.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "frame_list.json").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from road_anomaly import (
    binary_mask,
    discover_frames,
    load_semantic_mask,
    overlay_anomaly,
    records_to_frame,
)

records = discover_frames(ROOT)
inventory = records_to_frame(records)

label_values = sorted({value for record in records for value in record.label_values})
pixel_share = inventory["anomaly_pixels"].sum() / inventory["total_pixels"].sum()

print(f"Paired images: {len(records)}")
print(f"Resolution: {inventory['width'].min()} x {inventory['height'].min()} for every frame")
print(f"Distinct resolutions: {inventory[['width', 'height']].drop_duplicates().shape[0]}")
print(f"Mask values found: {label_values}")
print(f"Obstacle share of all pixels: {pixel_share:.1%}")
print(f"Median obstacle share of one image: {inventory['anomaly_ratio'].median():.1%}")
print(f"Smallest obstacle: {inventory['anomaly_ratio'].min():.2%}  ({inventory.loc[inventory['anomaly_ratio'].idxmin(), 'frame_id']})")
print(f"Largest obstacle:  {inventory['anomaly_ratio'].max():.1%}  ({inventory.loc[inventory['anomaly_ratio'].idxmax(), 'frame_id']})")
print(f"Instance masks present: {int(inventory['has_instance_mask'].sum())} / {len(inventory)}")
print(f"Instance masks that disagree with the semantic mask: {int((inventory['instance_agrees_with_semantic'] == False).sum())}")
print()
print(inventory.groupby("category").size().rename("images").to_string())

### Image and mask, side by side

The cell below draws one frame three ways: the photograph, the official mask (0 shown as black, 2 shown as red), and the mask placed on the photograph. The shapes printed above the figure must be identical in height and width. That is the correspondence check on a single example. The same check has already been applied to all 60 frames inside `discover_frames`.

In [ ]:
example_id = "animals17_E6_near_Kvaenangen"
example = inventory.loc[inventory["frame_id"] == example_id].iloc[0]
image = np.array(Image.open(example.image_path).convert("RGB"))
semantic = load_semantic_mask(Path(example.semantic_path))
mask = np.array(binary_mask(semantic))

print(f"{example_id}")
print(f"image {image.shape}    mask {semantic.shape}    official values {np.unique(semantic).tolist()}")
print(f"obstacle pixels: {int((semantic == 2).sum())} ({example.anomaly_ratio:.1%} of the frame)")

display_mask = np.stack([mask, np.zeros_like(mask), np.zeros_like(mask)], axis=-1)
figure, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
axes[0].imshow(image)
axes[0].set_title("Photograph")
axes[1].imshow(display_mask)
axes[1].set_title("Mask: red = obstacle")
axes[2].imshow(overlay_anomaly(image, mask))
axes[2].set_title("Mask placed on the photograph")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

### Figures for the report

These files are written by the preparation script. Read them in order with the captions in the report.

- **01** Half the images are animals. Cones, tires, and unusual vehicles are rare, so the split has to place them on purpose.
- **02** Every frame is 1280×720. What changes is the obstacle's area. The median image is 6.6% obstacle.
- **03** Cones and loose tires are small. Animals and rocks cover a wide range.
- **04** 90.2% of pixels are background. This is why accuracy is a weak score.
- **05** A typical scene from each category. The mask sits on the object.
- **06** The easiest and hardest sizes: 0.37% and 42.3% of the frame.
- **10** The same tire video, the same cardboard sculpture, and the same rockfall. These stay in training.
- **08** Letterbox to 512×512. The road is not stretched. Black bands are padding, and they contain no obstacle.
- **07** 42 training images, 9 validation, 9 test.
- **09** Every prepared scene. Green border = test, orange = validation, blue = training.

In [ ]:
from IPython.display import Image as ShowImage
from IPython.display import Markdown, display

figure_names = [
    "01_category_counts.png",
    "02_resolution_and_obstacle_size.png",
    "03_anomaly_area_by_category.png",
    "04_pixel_class_balance.png",
    "11_mask_correspondence_zoom.png",
    "05_image_mask_overlay_examples.png",
    "06_small_and_large_anomalies.png",
    "10_repeated_scenes.png",
    "08_preprocessing_letterbox.png",
    "07_split_composition.png",
    "09_all_scenes_by_split.png",
]
for name in figure_names:
    display(Markdown(f"**{name}**"))
    display(ShowImage(filename=str(ROOT / "reports" / "figures" / name)))

## 4. Data preparation

Three decisions, each with a reason:

1. **Binary mask.** Value 2 becomes 255. Training treats any pixel above 0 as the obstacle. There is no third class.
2. **Letterbox to 512×512.** The scale is 512/1280 = 0.4, so the picture becomes 512×288 and sits between two black bands of 112 pixels. Nearest-neighbour resizing keeps the mask discrete. Bilinear resizing is used for the photograph. The side 512 is divisible by 32, which a U-Net with five pooling stages needs. The original height 720 is not divisible by 32. Stretching to a square would distort the road, so we pad instead.
3. **Split 42 / 9 / 9.** The 11 frames in Figure 10 are one video or one photo sequence each, and they all go to training. The other frames are divided by filename category. Inside a category they are ranked by obstacle area, hold-out slots are spaced along that ranking, and the slots alternate between test and validation. Lost tires therefore do not appear in the test score. That is stated here so it is not a surprise when results are written up.

The next cell reads the manifest and repeats the checks.

In [ ]:
manifest = pd.read_csv(ROOT / "data" / "processed" / "split_manifest.csv")

print("Images per split and category")
print(pd.crosstab(manifest["split"], manifest["category"]).reindex(index=["train", "val", "test"]))
print()
print("Obstacle area in the original image (%)")
print((manifest.groupby("split")["anomaly_ratio_original"].agg(["min", "median", "max"]) * 100).round(2).reindex(["train", "val", "test"]))

groups_that_cross = int(manifest.groupby("scene_group")["split"].nunique().gt(1).sum())
smallest_prepared = int(manifest["anomaly_pixels_processed"].min())
assert groups_that_cross == 0, "A repeated scene was split across train and test"
assert smallest_prepared > 0, "Resizing deleted an obstacle"
assert list(manifest.groupby("split").size().reindex(["train", "val", "test"])) == [42, 9, 9]
assert set(manifest["canvas_size"]) == {512}

print()
print(f"Scene groups found in more than one split: {groups_that_cross}")
print(f"Smallest obstacle after resizing: {smallest_prepared} pixels")
print("Checks passed: 42/9/9, no repeated scene crosses a split, every obstacle survives the resize.")

### Hold-out images

Validation is used to watch training. Test is used once, for the final numbers. Both contain animals, one obstacle scene, two cone scenes, and one unusual vehicle, with small and large obstacles in each.

In [ ]:
holdout = manifest.loc[manifest["split"].isin(["val", "test"])].copy()
holdout["obstacle_area_%"] = (holdout["anomaly_ratio_original"] * 100).round(2)
columns = ["split", "category", "frame_id", "obstacle_area_%", "n_instances"]
holdout.sort_values(["split", "category", "frame_id"])[columns].reset_index(drop=True)

## 5. What this stage delivers

| Output | Where |
| --- | --- |
| This explanation, with the figures | `reports/01_business_and_data_understanding.md` |
| Per-image audit | `reports/tables/frame_inventory.csv` |
| Split assignment | `reports/tables/split_assignment.csv` and `data/processed/split_manifest.csv` |
| Network-ready images and masks | `data/processed/train`, `val`, and `test` |

Modelling has not started. The appropriate next model is a segmentation network, with transfer learning, because 42 images will not carry a deep network trained from scratch. Report precision, recall, F1, IoU, and Dice. Say explicitly that loose tires were kept in training, so they are a visual check rather than a test-set number.

To rebuild every file from the original `frames` folder:

```bash
python scripts/run_exploration_and_split.py
```